In [72]:
import pandas as pd

In [73]:
from tqdm import tqdm
tqdm.pandas()

In [74]:
import numpy as np

In [75]:
import boto3
bedrock = boto3.client(service_name="bedrock-runtime")
import json

In [76]:
import re
from time import time
from time import sleep

def extract_text_between_out_tags(input_string):
    # Use regular expression to find text between <out> tags
    pattern = r'<out>(.*?)</out>'
    matches = re.findall(pattern, input_string)
    
    # Return the extracted text
    return matches[0] if matches else None

def try_until_success(f, *args, **kwargs):
    while True:
        try:
            return f(*args, **kwargs)
        except:
            sleep(1)

def llm_claudev3(prompt, text):
    messages = [{"role": "user", "content": text}]
    body = json.dumps(
        {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 30000,
            "system": prompt,
            "messages": messages,
            "temperature": 0,
            "top_p": 0
        }  
    ) 
    response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-sonnet-20240229-v1:0")
    response_body = json.loads(response.get('body').read())
    return response_body["content"][0]["text"]

def generate_queries(row):
    text = f"PT: {row['PT']}, Attribute: {row['Question String']}, Value: {row['Picker Option Strings']}."
    
    PROMPT = """
    You are an expert at creating search queries. Given a product type (PT) and a question (related to the product type) and an answer (for the question).
    You need to generate a search query which would be helpful in searching the corresponding product.
    For e.g.
    PT: PAINT, Question: Which type of Paint are you looking for?, Answer: Spray Paint
    output should be <out>spray paint</out>
    PT: PAINT, Question: Which type of Paint are you looking for?, Answer: Waterproofing
    output should be <out>waterproofing paint</out>
    PT: STORAGE_HOOK, Question: Which type of Hook are you looking for?, Answer: Over the door
    output should be <out>over the door storage hook</out>
    """
    
    output = try_until_success(llm_claudev3, PROMPT, text)
    
    return extract_text_between_out_tags(output)

In [77]:
def generate_picker_queries(row):
    text = f"PT: {row['PT']}, Attribute: {row['refinement_name']}, Value: {row['picker_name']}."
    
    PROMPT = """
    You are an expert at creating search queries. Given a product type (PT) and a product attribute (related to the product type) and a value (associated with that attribute).
    You need to generate a search query which would be helpful in searching the corresponding product.
    For e.g.
    PT: PRINTER, Attribute: output_type, Value: Colour
    output should be <out>colour printer</out>
    PT: ELECTRONIC_WIRE, Attribute: Electronic Cable Connector Gender, Value: Male-to-Female
    output should be <out>male to female connector cable</out>
    PT: KEYBOARDS, Attribute: Keyboard Included Components, Value: USB Cable
    output should be <out>usb keyboard</out>
    PT: BRA, Attribute: Women's International Size, Value: M
    output should be <out>m size bra</out>
    """
    
    output = try_until_success(llm_claudev3, PROMPT, text)
    
    return output#extract_text_between_out_tags(output)

In [78]:
def extract_text_between_output_tags(input_string):
    # Use regular expression to find text between <out> tags
    pattern = r'<output>(.*?)</output>'
    matches = re.findall(pattern, input_string)
    return matches

def generate_picker_queries_df(dataframe):
    
    temp = dataframe.copy(deep=True)
    temp['input'] = '<input>PT: ' + temp['PT'].astype(str) + ', Question Code: ' + temp['Question Code'].astype(str) + ', Question String: ' + \
                    temp['Question String'].astype(str) + ', Picker Option: ' + temp['Picker Option Strings'].astype(str) + ', Next Question Code: ' + temp['Next Question Code'].astype(str) + '</input>'
    temp_string = '\n'.join(temp['input'].tolist())
    
    PROMPT = """
    You specialize in creating search queries tailored for specific product types (PT), based on user questions and picker options. 
    Each query is structured to match the specifc product feature precisely, following a hierarchical sequence indicated by the Next Question Code field.
    
    Rules:
    1. Ensure the output is in <output> tags.
    2. Ensure there is only one output for each input.
    If Next Question Code is blank, then there is no follow-up question.
    3. Note that you need to use the Question information as well to construct the output. For e.g.,
    <input>PT: DOOR_CLOSER, Question Code: 2, Question String: What is the weight of your door?, Picker Option: >80 kg, Next Question Code: </input>
    <thinking>The query neends to be generated for a door closer which supports a door of weight more than 80kg. </thinking>
    <output>door closer for over 80kg door</output>
    <input>PT: TREADMILL, Question Code: 2, Question String: What is the maximum user capacity?, Picker Option:	Up to 250 lbs, Next Question Code: </input>
    <thinking>Treadmill is for users up to 250 lbs</thinking>
    <output>treadmill for up to 250 lbs users</output>
    <input>PT: MATTRESS, Question Code: 2, Question String: Which Mattress thickness is right for me?, Picker Option: Up to 8 inches, Next Question Code: </input>
    <output>up to 8 inch thick mattress</output>
    <input>PT: SPORT_BAT, Question Code: 1, Question String: Who is the batter?, Picker Option: Adult, Next Question Code: </input>
    <output>adult sports bat</output>
    3. Keep track of the Question Code and Next Question Code. 
    The questions are sequential, selecting a certain option leads to a certain follow-up question. 
    The query for the next question depends on the parent question (determined by the next question code).
    4. Cases when you shouldn't use previous information:
    4.1. If the parent entry of a given input has 2 or more children
    4.2. If the input has multiple parent entries. For example:
    <input>PT: LADDER, Question Code: 5, Question String: Please select the necessary features required, Picker Option: Anti-Slip steps, Next Question Code: </input>
    <thinking>The input has two parents (Question 1, Telescopic ladder) and (Question 1, Multipurpose Ladder), hence the output is based on PT, Question String and Picker Option</thinking>
    <output>anti-slip step ladder</output>
    5. Cases when you should use previous information:
    5.1. If the parent entry of a given input has only 1 child. For example:
    <input>PT: WALLPAPER, Question Code: 3, Question String: Which colors do you want to buy? Picker Option: Blue, Next Question Code: </input>
    <thinking>The input has a single parent (Question 1, Telescopic ladder) and (Question 1, Peel & Stick Foam Tiles), hence the output is based on previous output "peel and stick foam tile wallpaper", 
    and Question String and Picker Option</thinking>
    <output>blue peel and stick foam tile wallpaper</output>
    """
    
    TEMP_INP_1 = """
    <input>PT: FAUCET, Question Code: 1, Question String: Which kind of faucet are you looking for?, Picker Option: Kitchen Faucet, Next Question Code: 2,3</input>
    <input>PT: FAUCET, Question Code: 1, Question String: Which kind of faucet are you looking for, Picker Option: Bathroom Faucet, Next Question Code: 4</input>
    <input>PT: FAUCET, Question Code: 1, Question String: Which kind of faucet are you looking for, Picker Option: Bathtub Faucet, Next Question Code: 5</input>
    <input>PT: FAUCET, Question Code: 2, Question String: Which type of kitchen faucet do you prefer?, Picker Option: Pot Filler, Next Question Code: </input>
    <input>PT: FAUCET, Question Code: 2, Question String: Which type of kitchen faucet do you prefer?, Picker Option: Touch on, Next Question Code: </input>
    <input>PT: FAUCET, Question Code: 2, Question String: Which type of kitchen faucet do you prefer?, Picker Option: Touchless, Next Question Code: </input>
    <input>PT: FAUCET, Question Code: 3, Question String: Which type of mounting are you looking for?, Picker Option: Centerset, Next Question Code: </input>
    <input>PT: FAUCET, Question Code: 3, Question String: Which type of mounting are you looking for?, Picker Option: Single Hole, Next Question Code: </input>
    <input>PT: FAUCET, Question Code: 4, Question String: Which type of mounting are you looking for?, Picker Option: Centerset, Next Question Code: </input>
    <input>PT: FAUCET, Question Code: 4, Question String: Which type of mounting are you looking for?, Picker Option: Single Hole, Next Question Code: </input>
    <input>PT: FAUCET, Question Code: 5, Question String: Which type of mounting are you looking for?, Picker Option: Deck-mounted, Next Question Code: </input>
    <input>PT: FAUCET, Question Code: 5, Question String: Which type of mounting are you looking for?, Picker Option: Wall, Next Question Code: </input>
    """
    
    TEMP_OUTPUT_1 = """
    <input>PT: FAUCET, Question Code: 1, Question String: Which kind of faucet are you looking for?, Picker Option: Kitchen Faucet, Next Question Code: 2,3</input>
    <thinking>This is the first question, so we need to create the query based on the PT, Question String and Picker Option.</thinking>
    <output>kitchen faucet</output>
    <input>PT: FAUCET, Question Code: 1, Question String: Which kind of faucet are you looking for, Picker Option: Bathroom Faucet, Next Question Code: 4</input>
    <output>bathroom faucet</output>
    <input>PT: FAUCET, Question Code: 1, Question String: Which kind of faucet are you looking for, Picker Option: Bathtub Faucet, Next Question Code: 5</input>
    <output>bathrub faucet</output>
    <input>PT: FAUCET, Question Code: 2, Question String: Which type of kitchen faucet do you prefer?, Picker Option: Pot Filler, Next Question Code: </input>
    <thinking>This is the second question, question 2 and 3 can be reached if 'Kitchen Faucet' is selected for question 1. Since the parent question (1, Kitchen Faucet), has more than one
    children - question 2 and question 3, we will generate the query based on the PT, Question String and Picker Option</thinking>
    <output>pot filler faucet</output>
    <input>PT: FAUCET, Question Code: 2, Question String: Which type of kitchen faucet do you prefer?, Picker Option: Touch on, Next Question Code: </input>
    <output>touch on faucet</output>
    <input>PT: FAUCET, Question Code: 2, Question String: Which type of kitchen faucet do you prefer?, Picker Option: Touchless, Next Question Code: </input>
    <output>touchless faucet</output>
    <input>PT: FAUCET, Question Code: 3, Question String: Which type of mounting are you looking for?, Picker Option: Centerset, Next Question Code: </input>
    <thinking>This is the third question, the parent of question 3 is question is question 1 option Kitchen Faucet. Since the parent question (1, Kitchen Faucet), has
    more than one children - question 2 and question 3, we will generate the query based on the PT, Question String and Picker Option</thinking>
    <output>centerset faucet</output>
    <input>PT: FAUCET, Question Code: 3, Question String: Which type of mounting are you looking for?, Picker Option: Single Hole, Next Question Code: </input>
    <output>single hole faucet</output>
    <input>PT: FAUCET, Question Code: 4, Question String: Which type of mounting are you looking for?, Picker Option: Centerset, Next Question Code: </input>
    <thinking>This is the fourth question, the parent of question 4 is question 1 option: Bathroom Faucet, Since the parent question (1, Bathroom Faucet), has only one child,
    we will use the PT, Question String, Picker Option and parent Option (viz., Bathroom Faucet) to generate the query.</thinking>
    <output>centerset bathroom faucet</output>
    <input>PT: FAUCET, Question Code: 4, Question String: Which type of mounting are you looking for?, Picker Option: Single Hole, Next Question Code: </input>
    <output>single hole bathroom faucet</output>
    <input>PT: FAUCET, Question Code: 5, Question String: Which type of mounting are you looking for?, Picker Option: Deck-mounted, Next Question Code: </input>
    <thinking>This is the fifth question, the parent of question 5 is question 1 option: Bathtub Faucet. Since the parent question (1, Bathtub Faucet) has only one child.
    we will use the PT, Question String, Picker Option and parent Option (viz., Bathtub Faucet) to generate the query.</thinking>
    <output>deck mounted bathtub faucet</output>
    <input>PT: FAUCET, Question Code: 5, Question String: Which type of mounting are you looking for?, Picker Option: Wall, Next Question Code: </input>
    <thinking>This is the fifth question, the parent of question 5 is question 1 option: Bathtub Faucet. Sine the parent question (1, Bathtub Faucet) has only one child
    we will use the PT, Question String, Picker Option and parent Option (viz., Bathtub Faucet) to generate: "wall mounted bathtub faucet", note the use of "mounted"
    since, the question was asking about the mounting.</thinking>
    <output>wall mounted bathtub faucet</output>
    """
    
    TEMP_INP_2 = """
    <input>PT: WALLPAPER, Question Code: 1, Question String: Which type of wallpaper do you want to buy?, Picker Option: Peel & Stick Wall Paper, Next Question Code: 2</input>
    <input>PT: WALLPAPER, Question Code: 1, Question String: Which type of wallpaper do you want to buy?, Picker Option: Peel & Stick Foam Tiles, Next Question Code: 3</input>
    <input>PT: WALLPAPER, Question Code: 1, Question String: Which type of wallpaper do you want to buy?, Picker Option: Premium Non Adhesive Wallpaper, Next Question Code: 2</input>
    <input>PT: WALLPAPER, Question Code: 2, Question String: Which designs do you want to buy?, Picker Option: Floral, Next Question Code: </input>
    <input>PT: WALLPAPER, Question Code: 2, Question String: Which designs do you want to buy?, Picker Option: Geometric, Next Question Code: </input>
    <input>PT: WALLPAPER, Question Code: 3, Question String: Which colors do you want to buy?, Picker Option: White, Next Question Code: </input>
    <input>PT: WALLPAPER, Question Code: 3, Question String: Which colors do you want to buy?, Picker Option: Grey, Next Question Code: </input>
    """
    
    TEMP_OUTPUT_2 = """
    <input>PT: WALLPAPER, Question Code: 1, Question String: Which type of wallpaper do you want to buy?, Picker Option: Peel & Stick Wall Paper, Next Question Code: 2</input>
    <output>peel and stick wallpaper</output>
    <input>PT: WALLPAPER, Question Code: 1, Question String: Which type of wallpaper do you want to buy?, Picker Option: Peel & Stick Foam Tiles, Next Question Code: 3</input>
    <output>peel and stick foam tile wallpaper</output>
    <input>PT: WALLPAPER, Question Code: 1, Question String: Which type of wallpaper do you want to buy?, Picker Option: Premium Non Adhesive Wallpaper, Next Question Code: 2</input>
    <output>premium non adhesive wallpaper</output>
    <input>PT: WALLPAPER, Question Code: 2, Question String: Which designs do you want to buy?, Picker Option: Floral, Next Question Code: </input>
    <thinking>This question has a two parents (Question 1, Peel & Stick Wall Paper) and (Question 1, Premium Non Adhesive Wallpaper), so we use PT, Question String, Picker Option to generate the output.</thinking>
    <output>floral wallpaper</output>
    <input>PT: WALLPAPER, Question Code: 2, Question String: Which designs do you want to buy?, Picker Option: Geometric, Next Question Code: </input>
    <output>geometric wallpaper</output>
    <input>PT: WALLPAPER, Question Code: 3, Question String: Which colors do you want to buy?, Picker Option: White, Next Question Code: </input>
    <thinking>This question has a single parent (Question 1, Peel & Stick Foam Tiles), so we use the previous output along with the current question details.</thinking>
    <output>white peel and stick foam tile wallpaper</output>
    <input>PT: WALLPAPER, Question Code: 3, Question String: Which colors do you want to buy?, Picker Option: Grey, Next Question Code: </input>
    <output>grey peel and stick foam tile wallpaper</output>
    """
    
    for retry in range(5):
        try:
            messages = [{"role": "user", "content": "Now try for this input:\n" + TEMP_INP_1 + \
                         "\nRemember to ensure that there exactly one output for every input."}, 
                        {"role": "assistant", "content": TEMP_OUTPUT_1}, 
                        {"role": "user", "content": "Great! Now try for this input:\n" + TEMP_INP_2  + \
                         "\nRemember to ensure that there exactly one output for every input."}, 
                        {"role": "assistant", "content": TEMP_OUTPUT_2},  
                        {"role": "user", "content": "Great job! Now try for this input:\n" + temp_string + \
                         "\nRemember to ensure that there exactly one output for every input."}]
            body = json.dumps(
                {
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 100000,
                    "system": PROMPT,
                    "messages": messages,
                    "temperature": 0,
                    "top_p": 0
                }  
            ) 
            response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-5-sonnet-20240620-v1:0")
            # response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-haiku-20240307-v1:0")
            response_body = json.loads(response.get('body').read())
            return response_body["content"][0]["text"]
        except Exception as e:
            print(e)
            sleep(1)

In [79]:
workflow = pd.read_excel("workflows/EU_final_workflows_new (3).xlsx")

In [80]:
workflow

,product_type,Legal Risk,Correct,topic_class,Question_check,Option_check,PT&Option,Keyword,Thumbnail Image Required (Y/N),Final Thumbnail Image,...,educational_content,educational_content_DE,educational_content_FR,educational_content_ES,educational_content_IT,educational_content_PT,topic_PT,Next_Q_ID_new,option_codes_new,preselect_new
0,CHAIR,NaN,question+option code+question code to be regen...,Application,0.0,NaN,CHAIROffice,office chair,N,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Propósito de uso da cadeira,"2,3",1#1,NaN
1,CHAIR,NaN,question+option code+question code to be regen...,Application,0.0,NaN,CHAIRGaming,gaming chair,N,https://m.media-amazon.com/images/I/41-MrksIqs...,...,NaN,NaN,NaN,NaN,NaN,NaN,Propósito de uso da cadeira,"2,3",1#2,NaN
2,CHAIR,NaN,question+option code+question code to be regen...,Application,0.0,NaN,CHAIRDining,dining chair,N,https://m.media-amazon.com/images/I/411nE81wBY...,...,NaN,NaN,NaN,NaN,NaN,NaN,Propósito de uso da cadeira,"2,3",1#3,NaN
3,CHAIR,NaN,question+option code+question code to be regen...,Application,0.0,NaN,CHAIRLiving room,living room chair,N,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Propósito de uso da cadeira,"2,3",1#4,NaN
4,CHAIR,NaN,question+option code+question code to be regen...,Application,0.0,NaN,CHAIRBedroom,bedroom chair,N,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Propósito de uso da cadeira,"2,3",1#5,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5685,ELECTRONIC_COMPONENT_FAN,NaN,Correct workflows,Attribute,0.0,NaN,ELECTRONIC_COMPONENT_FAN220.0 volts,220v electronic component fan,N,NaN,...,Features: Common high voltage AC rating.\nBene...,Merkmale: Gängige Hochspannungs-Wechselstrom-N...,Caractéristiques : Tension secteur courante él...,Características: Clasificación común de CA de ...,Caratteristiche: Comune tensione CA ad alta te...,Características: Classificação comum de alta t...,Voltagem,NaN,5#5,NaN
5686,ELECTRONIC_COMPONENT_FAN,NaN,Correct workflows,Attribute,0.0,NaN,ELECTRONIC_COMPONENT_FAN<3 watts,under 3 watt electronic component fan,N,NaN,...,"Features: Compact size, low noise, minimal pow...","Merkmale: Kompakte Größe, geringe Geräuschentw...","Caractéristiques : Taille compacte, faible bru...","Características: Tamaño compacto, bajo ruido, ...","Caratteristiche: Dimensioni compatte, basso ru...","Características: Tamanho compacto, baixo ruído...",Potência,NaN,6#1,"5#1, 5#3, 5#5"
5687,ELECTRONIC_COMPONENT_FAN,NaN,Correct workflows,Attribute,0.0,NaN,ELECTRONIC_COMPONENT_FAN<350 watts,under 350 watt electronic component fan,N,NaN,...,Features: Moderate airflow capacity.\nBenefits...,Merkmale: Mittlere Luftstromkapazität. \nVort...,Caractéristiques : Capacité d'écoulement d'air...,Características: Capacidad de flujo de aire mo...,Caratteristiche: Capacità di flusso d'aria mod...,Características: Capacidade de fluxo de ar mod...,Potência,NaN,6#2,"5#1, 5#3, 5#4, 5#5"
5688,ELECTRONIC_COMPONENT_FAN,NaN,Correct workflows,Attribute,0.0,NaN,ELECTRONIC_COMPONENT_FAN<2600 watts,under 2600 watt electronic component fan,N,NaN,...,Features: High-volume airflow capacity.\nBenef...,Merkmale: Hohe Luftstromkapazität. \nVorteile...,Caractéristiques : Capacité de flux d'air à gr...,Características: Capacidad de flujo de aire de...,Caratteristiche: Capacità di flusso d'aria ad ...,Características: Capacidade de fluxo de ar de ...,Potência,NaN,6#3,NaN


In [81]:
workflow.columns

Index(['product_type', 'Legal Risk', 'Correct', 'topic_class',
       'Question_check', 'Option_check', 'PT&Option', 'Keyword',
       'Thumbnail Image Required (Y/N)', 'Final Thumbnail Image',
       'Thumbnail Image option', 'is_multiselect', 'Q_ID_new', 'topic',
       'topic_DE', 'topic_FR', 'topic_ES', 'topic_IT', 'Question',
       'Question_DE', 'Question_FR', 'Question_ES', 'Question_IT',
       'Question_PT', 'options', 'options_DE', 'options_FR', 'options_ES',
       'options_IT', 'options_PT', 'educational_content',
       'educational_content_DE', 'educational_content_FR',
       'educational_content_ES', 'educational_content_IT',
       'educational_content_PT', 'topic_PT', 'Next_Q_ID_new',
       'option_codes_new', 'preselect_new'],
      dtype='object')

In [82]:
workflow = workflow.rename(columns={'product_type' : 'PT', 
                                    'Q_ID_new' : 'Question Code', 
                                    'Question_IT' : 'Question String', 
                                    'options_IT' : 'Picker Option Strings', 
                                    'Next_Q_ID_new' : 'Next Question Code'})

In [83]:
workflow['Next Question Code'] = workflow['Next Question Code'].fillna('')

In [84]:
def clean_value(value):
    # Remove spaces from numbers less than 10 and ensure no spaces if comma-separated
    if ',' in value:
        return ','.join([num.strip() for num in value.split(',') if num.strip().isnumeric()])
    elif value.isdigit() and int(value) < 10:
        return value.strip()
    
    # Convert non-numeric and numbers greater than 15 to ''
    if not value.isnumeric() or int(value) > 15:
        return ''

    return value

In [85]:
workflow = workflow[workflow['PT'].isin(['TELEVISION', 'CELLULAR_PHONE'])].dropna(subset=['Question Code']).reset_index(drop=True)

In [86]:
workflow['Question Code'] = workflow['Question Code'].apply(lambda x: str(int(x)))

In [87]:
workflow['PT'].value_counts()

PT
CELLULAR_PHONE    54
TELEVISION        48
Name: count, dtype: int64

In [88]:
workflow['generated_keyword'] = None

In [90]:
for pt in tqdm(workflow['PT'].unique()):
    print("PT: ", pt)
    temp = workflow[(workflow['PT'] == pt)]
    output = generate_picker_queries_df(temp)
    extracted_text = extract_text_between_output_tags(output)
    if len(extracted_text) == len(temp):
        workflow.loc[(workflow['PT'] == pt), 'generated_keyword'] = extracted_text
    else:
        print("Single batch generation failed for:", pt)

  0%|          | 0/2 [00:00<?, ?it/s]

PT:  CELLULAR_PHONE


 50%|█████     | 1/2 [00:27<00:27, 27.98s/it]

Single batch generation failed for: CELLULAR_PHONE
PT:  TELEVISION


100%|██████████| 2/2 [00:50<00:00, 25.13s/it]


In [94]:
for pt in tqdm(workflow.loc[workflow['generated_keyword'].isna(), 'PT'].unique()):
    print("PT: ", pt)
    
    # Filter for the current PT
    temp = workflow[workflow['PT'] == pt]
    
    # Get indices of NaN in 'generated_keyword' for the current PT
    nan_indices = temp[temp['generated_keyword'].isna()].index.tolist()
    
    # Process in sub-batches of 10
    for i in tqdm(range(0, len(nan_indices), 10)):
        sub_batch_indices = nan_indices[i:min(i + 10, len(nan_indices))]
        sub_temp = temp.loc[sub_batch_indices]

        output = generate_picker_queries_df(sub_temp)
        extracted_text = extract_text_between_output_tags(output)

        # Ensure the length of extracted_text matches sub_temp
        if len(extracted_text) == len(sub_temp):
            workflow.loc[sub_batch_indices, 'generated_keyword'] = extracted_text
        else:
            print(f"Mismatch for PT {pt}: {len(sub_temp)} vs {len(extracted_text)}")

0it [00:00, ?it/s]


In [97]:
workflow[['generated_keyword']].rename(
    columns={'generated_keyword' : 'keywords'}).to_csv(
    "qu_in_v2/IT_workflow_keywords.csv", index=False)

In [99]:
def generate_reformulation_df(dataframe):
    temp = dataframe.copy(deep=True)
    temp['input'] = '<input>PT: ' + temp['PT'].astype(str) + ', Code: ' + temp['Question Code'].astype(str) + \
    ', Query: ' + temp['generated_keyword'].astype(str) + ', Next Code: ' + temp['Next Question Code'].astype(str) + '</input>'
    temp_string = '\n'.join(temp['input'].tolist())
    
    PROMPT = """
    You are an expert at creating search queries. Your task is to process a sequence of input queries, removing specified words, and generating corresponding output tags. 
    Each input provides information including a Question Code and potentially a Next Question Code for sequential navigation.

    Rules:
    1. Ensure the output is in <output> tags.
    2. Ensure there is an output for each input.
    3. Maintain sequential tracking using Question Code and Next Question Code. If Next Question Code is empty, no follow-up question exists.
    4. If the parent entry has multiple codes in the Next Code field, then you need to subtract the product type from the current entry's "Query".
    E.g., we subtracted "faucet" from "pot filler faucet" to get "pot filler".
    5. If multiple parents lead to the same entry then again you need to subtract the product type from the current entry's "Query".
    6. If the parent entry has a single node in the Next Code field, then you need to subtract the "Query" of parent node from current entry's "Query".
    E.g., we subtracted "bathroom faucet" from "bathroom faucet for vessels" to get "for vessels".
    E.g., we subtracted "bathroom faucet for vessels" from "black bathroom faucet for vessels" to get "black".
    7. As a result of this process - post question code 1, none of the outputs should have the product type mentioned. If they do, then you are doing something wrong.
    In example I provided, none of the outputs post Question Code 1 has the term faucet mentioned.
    8. Here are some examples of subtraction: 
    "black aldrop lock" - "aldrop lock" = "black"
    "towerbolt lock for bathroom door" - "bathroom door lock" = "towerbolt"
    "bathroom faucet for vessels" - "bathroom faucet" = "for vessels"
    "digital door lock for bedroom door" - "bedroom lock" = "digital"
    "vessel bathroom faucet" - "bathroom faucet" = "vessel"
    """
    
    TEMP_INP = """
    <input>PT: FAUCET, Code: 1, Query: kitchen faucet, Next Code: 2,3</input>
    <input>PT: FAUCET, Code: 1, Query: bathroom faucet, Next Code: 4</input>
    <input>PT: FAUCET, Code: 2, Query: pot filler faucet, Next Code: </input>
    <input>PT: FAUCET, Code: 3, Query: centerset faucet, Next Code: </input>
    <input>PT: FAUCET, Code: 4, Query: bathroom faucet for vessels, Next Code: 5</input>
    <input>PT: FAUCET, Code: 5, Query: black bathroom faucet for vessels, Next Code: </input>
    """
    
    TEMP_OUTPUT = """
    <input>PT: FAUCET, Code: 1, Query: kitchen faucet, Next Code: 2,3</input>
    <thinking> This is the 1st question, we'll keep the query as is: kitchen faucet tap</thinking>
    <output>kitchen faucet</output>
    <input>PT: FAUCET, Code: 1, Query: bathroom faucet, Next Code: 4</input>
    <thinking> This is the 1st question, we'll keep the query as is: bathroom faucet tap</thinking>
    <output>bathroom faucet</output>  
    <input>PT: FAUCET, Code: 2, Query: pot filler faucet, Next Code: </input>
    <thinking>This is the 2nd question, the parent of this question is question 1 + "kitchen faucet". Both question 2 and 3 can be reached if 'kitchen faucet tap' is selected for question 1. 
    Since the parent question (1, kitchen faucet), has more than one children - question 2 and question 3, we will subtract the PT from the query.
    So, we'll remove "faucet" from "pot filler faucet", i.e., "pot filler"</thinking>
    <output>pot filler</output>
    <input>PT: FAUCET, Code: 3, Query: centerset faucet, Next Code: </input>
    <output>touch on</output>
    <input>PT: FAUCET, Code: 4, Query: bathroom faucet for vessels, Next Code: 5</input>
    <thinking>This is the 4th question, the parent of this question is question 1 + "bathroom faucet". Question 4 can only be reached if 'bathroom faucet' is selected in question 1.
    Since the parent question (1, bathroom faucet) has only one child - we will subtract the parent question query from this query.
    So, we'll remove "bathroom faucet" from "bathroom faucet for vessels", i.e., "for vessels"</thinking>
    <output>for vessels</output>
    <input>PT: FAUCET, Code: 5, Query: black bathroom faucet for vessels, Next Code: </input>
    <thinking>This is the 4th question, the parent of this question is question 4 + "bathroom faucet for vessels". Question 4 can only be reached if 'bathroom faucet for vessels' is selected in question 4.
    Since the parent question (4, bathroom faucet for vessels) has only one child - we will subtract the parent question query from this query.
    So, we'll remove "bathroom faucet for vessels" from "black bathroom faucet for vessels", i.e., "black"</thinking>
    <output>black</output>
    """
    
    while True:
        try:
            messages = [{"role": "user", "content": "Now try for this input:\n" + TEMP_INP}, 
                        {"role": "assistant", "content": TEMP_OUTPUT}, 
                        {"role": "user", "content": "Now try for this input:\n" + temp_string}]
            body = json.dumps(
                {
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 30000,
                    "system": PROMPT,
                    "messages": messages,
                    "temperature": 0,
                    "top_p": 0
                }  
            ) 
            # response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-sonnet-20240229-v1:0")
            # response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-haiku-20240307-v1:0")
            response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-5-sonnet-20240620-v1:0")
            response_body = json.loads(response.get('body').read())
            return response_body["content"][0]["text"]
        except:
            sleep(1)

In [100]:
for pt in tqdm(workflow['PT'].unique()):
    temp = workflow[(workflow['PT'] == pt)]
    output = generate_reformulation_df(temp)
    extracted_text = extract_text_between_output_tags(output)
    if len(extracted_text) == len(temp):
        workflow.loc[(workflow['PT'] == pt), 'Reformulation'] = extracted_text
    else:
        print("Single batch generation failed for:", pt)

100%|██████████| 2/2 [00:36<00:00, 18.06s/it]


In [ ]:
batch = 10

for pt in tqdm(workflow.loc[workflow['Reformulation'].isna(), 'PT'].unique()):
    temp = workflow[workflow['PT'] == pt]
    # Initialize an empty list to store extracted texts
    extracted_texts = []
    
    # Process each row in chunks of 10
    for i in range(0, len(temp), batch):
        # Calculate end index for the chunk
        end_idx = min(i + batch, len(temp))
        
        # Slice the dataframe to get the chunk
        temp_chunk = temp.iloc[i:end_idx]
        
        # Generate reformulation dataframe for the chunk
        output = generate_reformulation_df(temp_chunk)
        
        # Extract text between output tags
        extracted_text = extract_text_between_output_tags(output)
        
        # Append extracted text to the list
        extracted_texts.extend(extracted_text)
        
    if len(extracted_texts) == len(temp):
        # Assign extracted texts to Reformulation_ column in original dataframe
        workflow.loc[temp.index, 'Reformulation'] = extracted_texts
    else:
        print(len(extracted_texts), len(temp['generated_keyword']))
        print(extracted_texts)
        print(temp['generated_keyword'])
        # break

In [101]:
if 'index' not in workflow.columns:
    workflow = workflow.reset_index()

In [102]:
def generate_reformulation_df_(dataframe):
    temp = dataframe.copy(deep=True)
    temp['input'] = '<input>Attribute: ' + temp['Picker Option Strings'].astype(str) + ', Query: ' + temp['Reformulation'].astype(str) + '</input>'
    temp_string = '\n'.join(temp['input'].tolist())
    
    PROMPT = """
    You are an expert rectifying search queries.
    So given an attribute and search query - you need to ensure that the attribute is mentioned in the query.
    
    Rules:
    1. Ensure the output is in <output> tags.
    2. Ensure there is an output for every input.
    """
    
    TEMP_INP = """
    <input>Attribute: 30,100 BTU and above, Query: 30100 btu and above</input>
    <input>Attribute: RIM lock, Query: rim</input>
    <input>Attribute: Softball Bat, Query: softball</input>
    <input>Attribute: Up to 250 lbs, Query: for up to 250 lbs users</input>
    """
    
    TEMP_OUTPUT = """
    <input>Attribute: 30,100 BTU and above, Query: 30100 btu and above</input>
    <thinking> the attribute is mentioned in the query, we'll output the query as is: 30100 btu and above</thinking>
    <output>30100 btu and above</output>
    <input>Attribute: RIM lock, Query: rim</input>
    <thinking> the attribute isn't completely mentioned in the query, we'll modify it to contain the missing part: rim lock</thinking>
    <output>rim lock</output>
    <input>Attribute: Softball Bat, Query: softball</input>
    <thinking> the attribute isn't completely mentioned in the query, we'll modify it to contain the missing part: softball bat</thinking>
    <output>softball bat</output>
    <input>Attribute: Up to 250 lbs, Query: for up to 250 lbs users</input>
    <thinking>the attribute is mentioned in the query, we'll output the query as is: for up to 250 lbs users</thinking>
    <output>for up to 250 lbs users</output>
    """
    
    while True:
        try:
            messages = [{"role": "user", "content": "Now try for this input:\n" + TEMP_INP}, 
                        {"role": "assistant", "content": TEMP_OUTPUT}, 
                        {"role": "user", "content": "Great job! Now try for this input:\n" + temp_string}]
            body = json.dumps(
                {
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 30000,
                    "system": PROMPT,
                    "messages": messages,
                    "temperature": 0,
                    "top_p": 0
                }  
            ) 
            response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-sonnet-20240229-v1:0")
            # response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-haiku-20240307-v1:0")
            response_body = json.loads(response.get('body').read())
            return response_body["content"][0]["text"]
        except:
            sleep(1)

In [103]:
import time

In [104]:
for pt in tqdm(workflow['PT'].unique()):
    # Get all rows for the current PT
    temp = workflow[workflow['PT'] == pt]
    
    # Initialize an empty list to store extracted texts
    extracted_texts = []
    
    # Process each row in chunks of 10
    for i in range(0, len(temp), 10):
        # Calculate end index for the chunk
        end_idx = min(i + 10, len(temp))
        
        # Slice the dataframe to get the chunk
        temp_chunk = temp.iloc[i:end_idx]
        
        # Generate reformulation dataframe for the chunk
        output = generate_reformulation_df_(temp_chunk)
        
        # Extract text between output tags
        extracted_text = extract_text_between_output_tags(output)
        
        # Append extracted text to the list
        extracted_texts.extend(extracted_text)
        
    if len(extracted_texts) == len(temp):
        # Assign extracted texts to Reformulation_ column in original dataframe
        workflow.loc[temp.index, 'Reformulation_'] = extracted_texts
    else:
        print(extracted_texts)
        print(temp['Reformulation'])
        break

100%|██████████| 2/2 [00:47<00:00, 23.66s/it]


In [105]:
def generate_reformulation_final_df(dataframe):
    temp = dataframe.copy(deep=True)
    temp['input'] = '<input>PT: ' + temp['PT'].astype(str) + ', Query: ' + temp['Reformulation_'].astype(str) + '</input>'
    temp_string = '\n'.join(temp['input'].tolist())
    
    PROMPT = """
    You are an expert in product weights, sizes, etc on Amazon. 
    So given a product type (PT) and search query - if the query has any numerical range, you need to modify it, else keep the query as is.
    
    Rules:
    1. Ensure the output is in <output> tags.
    2. Ensure there is an output for every input.
    3. Ensure that you split queries only with numerical ranges while keeping the other queries untouched.
    """
    
    TEMP_INP = """
    <input>PT: DOOR_CLOSER, Query: for 40kg to 55kg door</input>
    <input>PT: DOOR_CLOSER,	Query: for over 80kg door</input>
    <input>PT: AIR_CONDITIONER, Query: up to 11899 BTU</input>
    <input>PT: PORTABLE_TOOL_BOX, Query: 1 to 3 drawers</input>
    <input>PT: PAINT, Query: grey spray paint</input>
    <input>PT: HARDWARE_HANDLE, Query: 12 inch</input>
    <input>PT: FAUCET, Query: single hole</input>
    <input>PT: TREADMILL, Query: for up to 300 lbs users</input>
    """
    
    TEMP_OUTPUT = """
    <input>PT: DOOR_CLOSER, Query: for 40kg to 55kg door</input>
    <thinking> This query has a numerical range, popular within this weight range are:
    45kg (99 lbs) - A standard weight for solid wood or fiberglass exterior entry doors.
    50kg (110 lbs) - Frequently used for larger entry doors, patio doors, or double doors.
    54kg (119 lbs) - A popular choice for heavy-duty exterior doors made of solid wood or steel.
    So the output should be: 40kg | 50kg | 54kg
    </thinking>
    <output>for 40kg door | for 50kg door | for 54kg door</output>
    
    <input>PT: DOOR_CLOSER,	Query: for over 80kg door</input>
    <thinking> This query has a numerical range, popular within this weight range are:
    80 kg (176 lbs): This weight is common for heavy-duty exterior doors made of solid wood or steel. It provides excellent security and soundproofing.
    90 kg (198 lbs): Doors in this weight range are often used for commercial applications, such as storefront doors or heavy-duty entry doors in public buildings.
    100 kg (220 lbs): This weight is typical for large commercial doors, such as those used in warehouses, industrial facilities, or loading docks.
    120 kg (264 lbs): Doors weighing around 120 kg are common in high-security applications, such as vault doors, blast-resistant doors, or specialized doors for data centers or secure facilities.
    So the output should be: 80kg | 90kg | 100kg | 120 kg
    </thinking>
    <output>for 80kg door | for 90kg door | for 100kg door | for 120 kg</output>
    
    <input>PT: AIR_CONDITIONER, Query: up to 11899 BTU</input>
    <thinking>This query has a numerical range, popular within this BTU range are:
    10000 BTU: This is a popular capacity for small to medium-sized rooms, providing efficient cooling for spaces up to approximately 450 square feet.
    11000 BTU: Slightly higher than 10,000 BTU units, these are also suitable for medium-sized rooms and provide effective cooling for areas up to around 500 square feet.
    11800 BTU: Close to the upper limit of your specified range, units of this capacity are designed for larger rooms or areas up to approximately 550 square feet.
    11899 BTU: Specifically at 11,899 BTU, this capacity would typically be found in units designed for larger rooms or more demanding cooling requirements, covering spaces up to around 550-600 square feet depending on the efficiency and specific design of the air conditioner.
    So the output should be: 10000 BTU | 11000 BTU | 11800 BTU | 11899 BTU
    </thinking>
    <output>10000 BTU | 11000 BTU | 11800 BTU | 11899 BTU</output>
    
    <input>PT: PORTABLE_TOOL_BOX, Query: 1 to 3 drawers</input>
    <thinking> This query has a numerical range, since the possible values here could be 1, 2 or 3, we'll simply create queries with all these possibilities.
    So the output should be: 1 drawer | 2 drawers | 3 drawers
    </thinking>
    <output>1 drawer | 2 drawers | 3 drawers</output>  
    
    <input>PT: PAINT, Query: grey spray paint</input>
    <thinking>This query doesn't have any numerical range, so we keep the query as is: grey spray paint</thinking>
    <output>grey spray paint</output>
    
    <input>PT: HARDWARE_HANDLE, Query: 12 inch</input>
    <thinking>This query has a number but doesn't have a numerical range, so we keep the query as is: 12 inch</thinking>
    <output>12 inch</output>
    
    <input>PT: FAUCET, Query: single hole</input>
    <thinking>This query doesn't have a numerical range, so we keep the query as is: single hole</thinking>
    <output>single hole</output>
    
    <input>PT: TREADMILL, Query: for up to 300 lbs users</input>
    <thinking>This query has a numerical range, popular values within the treadmill capacity range are:
    230 lbs (104 kg): This is a common weight capacity for entry-level or mid-range treadmills designed for home use.
    250 lbs (113 kg): Many residential treadmills have a weight capacity of 250 lbs, making them suitable for most users.
    300 lbs (136 kg): This is a popular weight capacity for higher-end home treadmills, as well as some light commercial models used in apartments, hotels, or small fitness centers.
    So the output should be: 230 lbs | 250 lbs | 300 lbs
    </thinking>
    <output>for 230 lbs users | for 250 lbs users | for 300 lbs users</output>
    """
    
    while True:
        try:
            messages = [{"role": "user", "content": "Now try for this input:\n" + TEMP_INP}, 
                        {"role": "assistant", "content": TEMP_OUTPUT}, 
                        {"role": "user", "content": "Now try for this input:\n" + temp_string}]
            body = json.dumps(
                {
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 30000,
                    "system": PROMPT,
                    "messages": messages,
                    "temperature": 0,
                    "top_p": 0
                }  
            ) 
            # response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-sonnet-20240229-v1:0")
            # response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-haiku-20240307-v1:0")
            response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-5-sonnet-20240620-v1:0")
            response_body = json.loads(response.get('body').read())
            return response_body["content"][0]["text"]
        except:
            sleep(1)

In [106]:
workflow = workflow.sample(frac=1, ignore_index=True)

In [107]:
workflow

,index,PT,Legal Risk,Correct,topic_class,Question_check,Option_check,PT&Option,Keyword,Thumbnail Image Required (Y/N),...,educational_content_ES,educational_content_IT,educational_content_PT,topic_PT,Next Question Code,option_codes_new,preselect_new,generated_keyword,Reformulation,Reformulation_
0,37,CELLULAR_PHONE,NaN,question+option code+question code to be regen...,Attribute,0.0,NaN,CELLULAR_PHONELow: <8Gb RAM,0,N,...,"Características: <8Gb RAM, 16-32Gb almacenamie...","Caratteristiche: <8Gb RAM, 16-32Gb memoria.\nV...","Características: <8Gb RAM, 16-32Gb armazenamen...",Memória RAM,,10#1,8#1,cellular phone with under 8GB RAM,under 8GB RAM,under 8GB RAM
1,23,CELLULAR_PHONE,NaN,question+option code+question code to be regen...,Attribute,0.0,NaN,CELLULAR_PHONEHigh capacity: 4000-5000mAh,0,N,...,Características: Tamaño de batería más grande ...,Caratteristiche: Dimensioni della batteria più...,Características: Tamanho da bateria maior do q...,Capacidade da bateria,,6#2,"3#1, 3#3",cellular phone with 4000-5000mAh battery,4000-5000mAh battery,4000-5000mAh battery
2,5,CELLULAR_PHONE,NaN,question+option code+question code to be regen...,Application,0.0,NaN,CELLULAR_PHONENavigation/maps,0,N,...,NaN,NaN,NaN,Uso principal do telefone,"2,3,4,5,6,7,8,9,10,11,12,13",1#6,NaN,telefono cellulare per navigazione e mappe,navigation and maps,navigation and maps
3,94,TELEVISION,NaN,Correct workflows,Attribute,0.0,NaN,TELEVISIONDolby Vision,Dolby Vision television,N,...,Características: Formato HDR propietario con m...,Caratteristiche: Formato HDR proprietario con ...,Características: Formato HDR proprietário com ...,Suporte HDR,,9#2,"4#1, 4#2, 4#3, 4#4, 4#5",Dolby Vision television,Dolby Vision,Dolby Vision
4,2,CELLULAR_PHONE,NaN,question+option code+question code to be regen...,Application,0.0,NaN,CELLULAR_PHONEMedia consumption,0,N,...,NaN,NaN,NaN,Uso principal do telefone,"2,3,4,5,6,7,8,9,10,11,12,13",1#3,NaN,telefono cellulare per social media,social media,social media
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,76,TELEVISION,NaN,Correct workflows,Attribute,0.0,NaN,TELEVISION100 hertz,100 hertz refresh rate television,N,...,Características: Actualiza la imagen 100 veces...,Caratteristiche: Aggiorna l'immagine 100 volte...,Características: Atualiza a imagem 100 vezes p...,Taxa de atualização,,5#5,NaN,100 hertz television,100 hertz,100 hertz
98,61,TELEVISION,NaN,Correct workflows,Sub-type,0.0,NaN,TELEVISIONQLED,qled tv,N,...,Características: Capa de retroiluminación de p...,Caratteristiche: Strato di retroilluminazione ...,Características: Camada de retroiluminação de ...,Tecnologia de exibição,,2#3,NaN,QLED television,QLED,QLED
99,74,TELEVISION,NaN,Correct workflows,Attribute,0.0,NaN,TELEVISIONover 144Hz,over 144Hz refresh rate television,N,...,Características: Tasas de actualización altas ...,Caratteristiche: Frequenze di aggiornamento el...,Características: Taxas de atualização altas ac...,Taxa de atualização,,5#3,NaN,over 144Hz television,over 144Hz,over 144Hz
100,92,TELEVISION,NaN,Correct workflows,Sub-type,0.0,NaN,TELEVISIONLCD,LCD television,N,...,Características: Tecnología de pantalla de cri...,Caratteristiche: Tecnologia a cristalli liquid...,Características: Tecnologia de tela de cristal...,Tecnologia de exibição,,8#4,"5#1, 5#4, 5#5",LCD television,LCD,LCD


In [109]:
for pt in tqdm(workflow['PT'].unique()):
    temp = workflow[workflow['PT'] == pt]
    # Initialize an empty list to store extracted texts
    extracted_texts = []
    
    # Process each row in chunks of 10
    for i in range(0, len(temp), 15):
        # Calculate end index for the chunk
        end_idx = min(i + 15, len(temp))
        
        # Slice the dataframe to get the chunk
        temp_chunk = temp.iloc[i:end_idx]
        
        # Generate reformulation dataframe for the chunk
        output = generate_reformulation_final_df(temp_chunk)
        
        # Extract text between output tags
        extracted_text = extract_text_between_output_tags(output)
        
        # Append extracted text to the list
        extracted_texts.extend(extracted_text)
        
    if len(extracted_texts) == len(temp):
        # Assign extracted texts to Reformulation_ column in original dataframe
        workflow.loc[temp.index, 'Reformulation_Final'] = extracted_texts
    else:
        print(extracted_texts)
        print(temp['Reformulation_'])
        break

100%|██████████| 2/2 [01:30<00:00, 45.08s/it]


In [110]:
workflow = workflow.sort_values('index', ignore_index=True)

In [112]:
workflow[['PT', 'Question Code', 'Question String', 'Picker Option Strings', 
          'generated_keyword', 'Reformulation', 'Reformulation_Final']].to_excel(
    'reformulation_validation_IT_v9.xlsx', index=False)

In [115]:
workflow_sample = workflow.reset_index(drop=True)

In [116]:
nodes = pd.read_parquet("s3://ajjagata/2024/data_dumps/o_browse_node/v0/phase_all/region_id=2/marketplace_id=623225021/")
nodes['browse_node_id'] = nodes['browse_node_id'].astype(str)
nodes['parent_browse_node_id'] = nodes['parent_browse_node_id'].astype(str)
nodes['edge_path_name'] = nodes['edge_path_name'].fillna('')
nodes['full_name'] = nodes['browse_root_name'].astype(str) + ':' + nodes['edge_path_name'].astype(str)
bn2name = dict(zip(nodes.browse_node_id, nodes.full_name))

In [117]:
nodes.browse_root_name.value_counts().head()

browse_root_name
eg-automated-refinements              163376
eg-refinements                         63617
eu-thesaurus-browse-generalization     32585
eg-fashion                              5316
eg-product-types                        3898
Name: count, dtype: int64

In [136]:
def get_keywords(x):
    return x['queryInfo'][0]['meta']['keywords']

def get_attribute(x, attribute):
    for i in x['queryInfo']:
        if ('type' in i) and (i['type'] == attribute):
            return i['meta']
        
def get_qu_keywords(file_path):
    keywords = []
    refinements = []
    browse_nodes = []
    pt = []

    with open(file_path, "r") as f:
        for ii, line in enumerate(tqdm(f)):
            output = json.loads(line)
            extracted_keywords = get_keywords(output)
            keywords.append(extracted_keywords)
            
    metadata = pd.DataFrame({'keywords' : keywords})
    return metadata

def get_qu_df(file_path):
    keywords = []
    refinements = []
    browse_nodes = []
    pt = []

    with open(file_path, "r") as f:
        for ii, line in enumerate(tqdm(f)):
            try:
            # if True:
                output = json.loads(line)
                extracted_keywords = get_keywords(output)
                extracted_refinements = ""#get_attribute(output, 'q2refinement')['refinement-scores']
                extracted_browse_nodes = get_attribute(output, 'qba')['browse-scores']
                # extracted_pt = get_attribute(output, 'q2pt')['q2pt'].split(':')[0]

                keywords.append(extracted_keywords)
                # pt.append(extracted_pt)
                refinements.append(extracted_refinements)        
                browse_nodes.append(extracted_browse_nodes)
            except:
                # pass
                output = json.loads(line)
                extracted_keywords = get_keywords(output)
                print(extracted_keywords)
    metadata = pd.DataFrame({'keywords' : keywords, 'refinements' : refinements, 'browse_nodes' : browse_nodes})
    return metadata

In [137]:
def get_qu_merged_df(file_paths):
    keywords = get_qu_keywords(file_paths[0])
    
    outputs = []
    for file_path in file_paths:
        outputs.append(get_qu_df(file_path))
    output_df = pd.concat(outputs).drop_duplicates(subset=['keywords'], ignore_index=True)
    return keywords.merge(output_df, on='keywords')

In [138]:
def text_to_dataframe(text):
    rows = text.split(",")
    data = []
    
    for row in rows:
        elements = row.split(":")
        data.append(elements)
    
    df = pd.DataFrame(data, columns=["browse_node", "depth", "parent_node", "score"])
    df['depth'] = df['depth'].astype(int)
    df['score'] = df['score'].astype(float)
    
    return df.sort_values('depth').sort_values('score', ascending=False, ignore_index=True, kind='merge')

In [139]:
def select_browse_node(df):
    # Filter depth 1 nodes
    root_nodes = df[df['depth'] == 1]
    
    # Check if there are multiple root nodes
    if len(root_nodes) > 1:
        # Choose the root node with the highest score
        chosen_node = root_nodes.loc[root_nodes['score'].idxmax()]
    else:
        # Only one root node
        chosen_node = root_nodes.iloc[0]

    # Check if the chosen node's score is above 98%
    if chosen_node['score'] >= 0.85:
        return chosen_node['browse_node']
    
    # Check for children nodes
    children_nodes = df[df['parent_node'] == chosen_node['browse_node']]
    
    # Check if there are any children nodes
    if not children_nodes.empty:
        # Sort children nodes by score
        children_nodes = children_nodes.sort_values(by='score', ascending=False)
        # Choose the child node with the highest score
        chosen_node = children_nodes.iloc[0]

    return chosen_node['browse_node']

In [140]:
def select_recommended_browse_node(df, similarity_browse_node):
    # Step 1: Find root nodes and their QBA scores
    root_nodes = df[df['depth'] == 1]
    
    if len(root_nodes) == 0:
        raise ValueError("No root nodes found in the dataframe.")
    
    # If there are multiple root nodes, choose the one with the higher QBA score
    if len(root_nodes) == 1:
        root_node = root_nodes.iloc[0]
    else:
        root_node = root_nodes.loc[root_nodes['score'].idxmax()]
    
    root_node_id = root_node['browse_node']
    root_node_qba = root_node['score']
    
    
    # Step 3: Check if similarity root ancestor score > 85% of root node score
    similarity_nodes = df[df['browse_node'].isin(similarity_browse_node)].reset_index(drop=True)
    similarity_node = similarity_nodes.iloc[similarity_nodes['score'].idxmax()]
    similarity_node_score = similarity_node['score']
    
    while (similarity_node['depth'] != 1):

        if (similarity_node_score / root_node_qba) > 0.85:
            return similarity_node['browse_node']
        
        similarity_nodes = df[df['browse_node'] == similarity_node['parent_node']].reset_index(drop=True)
        similarity_node = similarity_nodes.loc[0]
        similarity_node_score = similarity_node['score']
        
    # Step 4: Select child node with higher threshold for QBA score at 85%
    child_nodes = df[df['parent_node'] == root_node_id].reset_index(drop=True)
    
    if len(child_nodes) > 0:
        max_child_node = child_nodes.loc[child_nodes['score'].idxmax()]
        child_node_id = max_child_node['browse_node']
        child_node_qba = max_child_node['score']
        
        if child_node_qba / root_node_qba >= 0.85:
            return child_node_id
    
    # Step 5: Find the next child node without any other siblings
    if len(child_nodes) == 1:
        lone_child_node_id = child_nodes.loc[0, 'browse_node']
        return lone_child_node_id
    
    # Step 6: Use the root node if previous conditions are not satisfied
    return root_node_id

In [141]:
import os

def get_absolute_paths(directory):
    # Convert relative path to absolute path
    directory = os.path.abspath(directory)
    
    # Ensure the directory exists
    if not os.path.isdir(directory):
        raise ValueError(f"The directory {directory} does not exist or is not a directory.")
    
    # List to hold absolute paths
    absolute_paths = []

    # Walk through the directory
    for root, dirs, files in os.walk(directory):
        for file in files:
            # Join the root directory with the file name to get the full path
            full_path = os.path.join(root, file)
            absolute_paths.append(full_path)
    
    return absolute_paths


In [142]:
qu_df = get_qu_merged_df(get_absolute_paths('./qu_output/EG/'))

1377it [00:00, 10573.39it/s]
1377it [00:00, 10209.47it/s]

gaming chair
landscape camera lens
architectural speakers
drone with 300-500m operating range
underwater action camcorder
walking treadmill
ceiling-mounted electric fan
washer with spin cycle


In [145]:
def extract_text_between_output_tags(input_string):
    # Use regular expression to find text between <out> tags
    pattern = r'<output>(.*?)</output>'
    matches = re.findall(pattern, input_string)
    return matches


def get_similar_nodes(keyword, bn_recos):
    ids = [i.split(':')[0] for i in bn_recos.split(',')]
    names = '\n'.join([f"<category>{ii + 1}. " + bn2name.get(id_, id_) + "</category>" for ii, id_ in enumerate(ids)])
    temp_string = f"""<input><keyword>{keyword}</keyword>\n{names}</input>"""
    
    
    PROMPT = """
    You are an expert at computing text similarity. Given, a search keyword and a list of category names you
    are capable of identifying which category name matches the keyword most aptly.
    
    For e.g.
    <input>
    <keyword>regular dumbbells</keyword>
    <category>1. us-sporting-goods:</category>
    <category>2. us-sporting-goods:/Categories/Exercise & Fitness</category>
    <category>3. Categories/Exercise & Fitness/Strength Training Equipment</category>
    <category>4. Exercise & Fitness/Strength Training Equipment/Weights & Accessories</category>
    <category>5. Strength Training Equipment/Weights & Accessories/Dumbbells</category>
    </input>

    <thinking>The category which is most apt for "regular dumbbells" is "5. Strength Training Equipment/Weights & Accessories/Dumbbells".
    Since 1) largely the whole name of category seems relevant for the keyword.
    2) the last part of the category name "Dumbbells" is matching with the keyword.
    </thinking>
    <output>5</output>
    
    <input>
    <keyword>ergonomic office chair</keyword>
    <category>1. Categories/Office Furniture & Lighting/Chairs & Sofas</category>
    <category>2. Office Furniture & Lighting/Chairs & Sofas/Managerial & Executive Chairs</category>
    <category>4. us-home-garden:</category>
    <category>5. us-home-garden:/Categories/Furniture</category>
    <category>6. Home Office Furniture/Home Office Chairs/Home Office Desk Chairs</category>
    <category>7. us-office-products:</category>
    <category>9. us-office-products:/Categories/Office Furniture & Lighting</category>
    </input>
    
    <thinking>
    Multiple categories seem to be similar to this:
    "1. Categories/Office Furniture & Lighting/Chairs & Sofas"
    "2. Office Furniture & Lighting/Chairs & Sofas/Managerial & Executive Chairs"
    "6. Home Office Furniture/Home Office Chairs/Home Office Desk Chairs"
    </thinking>
    
    <output>1,2,6</output>
    
    Rules:
    1. Ensure the output is in <output> tags.
    """
    
    output = try_until_success(llm_claudev3, PROMPT, "Now try for this input:\n" + temp_string)
    
    return output 

def get_deepest_node_new(df):
    # Ensure 'depth' and 'score' columns are numeric for proper comparison
    df['depth'] = pd.to_numeric(df['depth'], errors='coerce')
    df['score'] = pd.to_numeric(df['score'], errors='coerce')
    
    # Find the maximum depth
    max_depth = df['depth'].max()
    
    # Filter nodes with the maximum depth
    deepest_nodes = df[df['depth'] == max_depth]
    
    # Find the node with the highest score at the deepest depth
    deepest_node = deepest_nodes.loc[deepest_nodes['score'].idxmax()]
    
    return deepest_node['browse_node']

def extract_col(logic):
    return extract_text_between_output_tags(logic)

def get_similar_nodes_(bn_recos, logic):
    try:
        node_ids = [int(ii.strip()) - 1 for ii in logic[0].split(',')]
        bn_ids = [i.split(':')[0] for i in bn_recos.split(',')]
        return [bn_ids[i] for i in node_ids]
    except:
        bn_ids = [get_deepest_node_new(text_to_dataframe(bn_recos))]
        return bn_ids

In [ ]:
qu_df['similar_node_logic'] = qu_df.progress_apply(lambda x: get_similar_nodes(x['keywords'], x['browse_nodes']), axis=1)

100%|██████████| 1369/1369 [1:30:20<00:00,  3.96s/it]


In [ ]:
qu_df['similar_node_ext'] = qu_df.progress_apply(lambda x: extract_col(x['similar_node_logic']), axis=1)

In [ ]:
qu_df['similar_node'] = qu_df.progress_apply(lambda x: get_similar_nodes_(x['browse_nodes'], x['similar_node_ext']), axis=1)

In [ ]:
qu_df['similar_node_name'] = qu_df['similar_node'].progress_apply(lambda x: [bn2name.get(i, i) for i in x])

In [ ]:
qu_df['recommended_node'] = qu_df.progress_apply(lambda x: select_recommended_browse_node(text_to_dataframe(x['browse_nodes']), 
                                                                                      x['similar_node']), axis=1)

In [ ]:
qu_df['recommended_node_name'] = qu_df['recommended_node'].apply(lambda x: bn2name.get(x, x))

In [ ]:
nodes = nodes.astype(str)
refinement2picker_nodes = nodes.groupby('parent_browse_node_id')['browse_node_id'].apply(list).to_dict()

In [ ]:
def check_refinement(node_id):
    node_name = bn2name.get(node_id, '').lower()
    for i in ['price', 'review', 'delivery', 'deals & discounts',  'condition',
              'new arrivals', 'eligible for free shipping', 'availability', 'site-wide refinements']:
        if i in node_name:
            return False
    return True

def get_pickers(refinement_string):
    if isinstance(refinement_string, str):
        refinements = [i.split(':')[0] for i in refinement_string.split('|') if float(i.split(':')[-1]) > 0.01]
        cleaned_refinements = [i for i in refinements if check_refinement(i)]
        picker_list = []
        for refinement in cleaned_refinements:
            picker_nodes = refinement2picker_nodes.get(refinement, [])
            if len(picker_nodes) > 0:
                picker_list += picker_nodes
        return picker_list
    else:
        return []

In [92]:
def pretty_bn_name(ids):
    name = bn2name.get(ids, ids)
    return '/'.join(name.split('/')[-3:])

def extract_text_between_output_tags(input_string):
    # Use regular expression to find text between <out> tags
    pattern = r'<output>(.*?)</output>'
    matches = re.findall(pattern, input_string)
    return matches

def get_similar_refinements(pt, que, attr, query):
    ids = [i.split('_')[-1] for i in query2refinement_dict.get(query, ['alpha']) if i.split('_')[-1].isnumeric() and check_refinement(i.split('_')[-1])]
    names = ', '.join(['0. None'] + [f"{ii + 1}. " + pretty_bn_name(id_) for ii, id_ in enumerate(ids)])
    temp_string = f"""<input>Que. Product: {pt.replace('_', ' ').lower()}. {que} - Attribute: {attr}\n{names}</input>"""

    PROMPT = """You are an expert shopping assistant, a customer is trying to buy a product on Amazon. The customer is looking for a particular attribute, you need to identify
    if there are any search filters on Amazon which almost exactly match the attribute the customer is looking for. There would be times when you aren't able to find a match (indiciated by the first option)
    and that is fine.

    Rules to keep in mind:
    1. Identify filters which exactly match the Attribute and Question.
    2. Don't select filters which could possibly be meant by the customer. Only select filters which are definitely meant by the customer.
    """
    
    TEMP_INP_1 = """
    <input>Que. Product: sport bat. What bat drop do you need? - Attribute: -12
    0. None, 1. Sporting Goods/Baseball & Softball Bat Length/24 Inch, 2. Sporting Goods/Baseball & Softball Bat Length/25 Inch, 3. Sporting Goods/Baseball & Softball Bat Length/26 Inch, 4. Sporting Goods/Baseball & Softball Bat Length/27 Inch, 5. Sporting Goods/Baseball & Softball Bat Length/28 Inch, 6. Sporting Goods/Baseball & Softball Bat Length/29 Inch, 7. Sporting Goods/Baseball & Softball Bat Length/30 Inch, 8. Sporting Goods/Baseball & Softball Bat Length/31 Inch, 9. Sporting Goods/Baseball & Softball Bat Length/32 Inch, 10. Sporting Goods/Baseball & Softball Bat Length/33 Inch, 11. Sporting Goods/Baseball & Softball Bat Length/34 Inch, 12. Sporting Goods/Baseball & Softball Bat Length/35 Inch, 13. Sporting Goods/Baseball & Softball Bat Length/36 Inch, 14. Sporting Goods/Baseball & Softball Bat Length/37 Inch, 15. Sporting Goods/Baseball & Softball Bat Material/Aluminum, 16. Sporting Goods/Baseball & Softball Bat Material/Composite, 17. Sporting Goods/Baseball & Softball Bat Material/Plastic, 18. Sporting Goods/Baseball & Softball Bat Material/Wood
    </input>
    """
    
    TEMP_OUT_1 = """
    <thinking>
    The attribute -12 drop isn't part of any of the search filters. Hence, the output should be - 0, corresponding to 0. None
    </thinking>
    <output>0</output>
    """
    
    TEMP_INP_2 = """
    <input>Que. Product: lock. Which type of lock you are looking for? - Attribute: Mortise Locks
    0. None, 1. HA/Hardware Handle Exterior Finish/Wood, 2. HA/Hardware Handle Exterior Finish/Bronze, 3. HA/Hardware Handle Exterior Finish/Brass, 4. HA/Hardware Handle Exterior Finish/Iron, 5. HA/Hardware Handle Exterior Finish/Chrome, 6. HA/Hardware Handle Exterior Finish/Stainless Steel, 7. HA/Hardware Handle Exterior Finish/Pewter, 8. HA/Hardware Handle Exterior Finish/Aluminum, 9. HA/Hardware Handle Exterior Finish/Nickel, 10. HA/Hardware Handle Exterior Finish/Copper, 11. HA/Hardware Handle Exterior Finish/Zinc
    </input>
    """
    
    TEMP_OUT_2 = """
    <thinking>
    The attribute mortise doesn't match with any of the search filters. Hence, the output should be - 0, corresponding to 0. None
    </thinking>
    <output>0</output>
    """
    
    TEMP_INP_3 = """
    <input>Que. Product: lock. Which finish are you looking for? - Attribute: Chrome
    0. None, 1. D/DIY_&_Tools-Color/Black, 2. D/DIY_&_Tools-Color/Grey, 3. D/DIY_&_Tools-Color/White, 4. D/DIY_&_Tools-Color/Brown, 5. D/DIY_&_Tools-Color/Beige, 6. D/DIY_&_Tools-Color/Red, 7. D/DIY_&_Tools-Color/Pink, 8. D/DIY_&_Tools-Color/Orange, 9. D/DIY_&_Tools-Color/Yellow, 10. D/DIY_&_Tools-Color/Ivory, 11. D/DIY_&_Tools-Color/Green, 12. D/DIY_&_Tools-Color/Blue, 13. D/DIY_&_Tools-Color/Purple, 14. D/DIY_&_Tools-Color/Gold, 15. D/DIY_&_Tools-Color/Silver, 16. D/DIY_&_Tools-Color/Multi, 17. D/DIY_&_Tools-Color/Clear, 18. D/DIY_&_Tools-Color/Stainless Steel, 19. L/Lighting-Color/Beige, 20. L/Lighting-Color/Black, 21. L/Lighting-Color/Blue, 22. L/Lighting-Color/Brass, 23. L/Lighting-Color/Bronze, 24. L/Lighting-Color/Brown, 25. L/Lighting-Color/Brushed Steel, 26. L/Lighting-Color/Clear, 27. L/Lighting-Color/Copper, 28. L/Lighting-Color/Gold, 29. L/Lighting-Color/Grey, 30. L/Lighting-Color/Green, 31. L/Lighting-Color/Iron, 32. L/Lighting-Color/Ivory, 33. L/Lighting-Color/Multi-Color, 34. L/Lighting-Color/Nickel, 35. L/Lighting-Color/Oil-Rubbed Bronze, 36. L/Lighting-Color/Orange, 37. L/Lighting-Color/Pewter, 38. L/Lighting-Color/Pink, 39. L/Lighting-Color/Purple, 40. L/Lighting-Color/Red, 41. L/Lighting-Color/Rust, 42. L/Lighting-Color/Silver, 43. L/Lighting-Color/Stainless-Steel, 44. L/Lighting-Color/White, 45. L/Lighting-Color/Yellow, 46. HA/Hardware Handle Exterior Finish/Wood, 47. HA/Hardware Handle Exterior Finish/Bronze, 48. HA/Hardware Handle Exterior Finish/Brass, 49. HA/Hardware Handle Exterior Finish/Iron, 50. HA/Hardware Handle Exterior Finish/Chrome, 51. HA/Hardware Handle Exterior Finish/Stainless Steel, 52. HA/Hardware Handle Exterior Finish/Pewter, 53. HA/Hardware Handle Exterior Finish/Aluminum
    </input>
    """
    
    TEMP_OUT_3 = """
    <thinking>
    The attribute chrome finish matches the search filter - 50, corresponding to 50. HA/Hardware Handle Exterior Finish/Chrome
    </thinking>
    <output>50</output>
    """
    
    
    while True:
        try:
            messages = [{"role": "user", "content": "Now try for this input:\n" + TEMP_INP_1}, 
                        {"role": "assistant", "content": TEMP_OUT_1}, 
                        {"role": "user", "content": "Good job! Now try for this input:\n" + TEMP_INP_2}, 
                        {"role": "assistant", "content": TEMP_OUT_2}, 
                        {"role": "user", "content": "Good job! Now try for this input:\n" + TEMP_INP_3}, 
                        {"role": "assistant", "content": TEMP_OUT_3}, 
                        {"role": "user", "content": "Good job! Now try for this input:\n" + temp_string}]
            body = json.dumps(
                {
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 30000,
                    "system": PROMPT,
                    "messages": messages,
                    "temperature": 0,
                    "top_p": 0
                }  
            ) 
            response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-sonnet-20240229-v1:0")
            response_body = json.loads(response.get('body').read())
            return response_body["content"][0]["text"]
        except:
            sleep(1)

In [91]:
def get_similar_refinements_(query, logic):
    ids = [i for i in query2refinement_dict.get(query, ['alpha']) if i.split('_')[-1].isnumeric() and check_refinement(i.split('_')[-1])]
    try:
        node_ids = [int(ii.strip()) - 1 for ii in extract_text_between_output_tags(logic)[0].split(',')]
    except:
        node_ids = []
        
    final_output = [ids[i].split('_')[-1] for i in node_ids if (i < len(ids)) and (i >= 0)]
    
    # final_output_ = [ids[i].split('_')[-2] for i in node_ids if (i < len(ids)) and (i >= 0)]
    
    if len(final_output) == 1:
        return final_output
    else:
        return []

In [ ]:
pd.set_option('display.max_colwidth', 0)

In [ ]:
required_qu = qu_df[['keywords', 'recommended_node', 'recommended_node_name', 'similar_node', 'similar_node_name', 'refinements']]
required_qu = required_qu.rename(columns={'keywords' : 'generated_keyword'})

In [ ]:
qu_df_temp = workflow_sample.merge(required_qu.drop_duplicates(subset=['generated_keyword']), on='generated_keyword', how='left')#.head(100)


In [90]:
qu_df_temp.columns

Index(['Unnamed: 0', 'index', 'PT', 'topic_class', 'Question Code',
       'Question String', 'Picker Option Strings', 'Next Question Code',
       'option_codes', 'preselect', 'is_multiselect', 'Question Arabic',
       'options Arabic', 'topic', 'topic Arabic', 'generated_keyword',
       'Reformulation', 'Reformulation_', 'Reformulation_Final',
       'recommended_node', 'recommended_node_name', 'similar_node',
       'similar_node_name', 'refinements'],
      dtype='object')

In [88]:
query2refinement = pd.read_csv("s3://mlblr/deepnsp/buying_guides/output_datasets/raw/EG/v3/Buying_guides_EG_preds_flan_t5_small_multi_head_cls_lim_all_topk_200_npa_ext_unk_split.csv")

In [ ]:
query2refinement_dict = dict(zip(query2refinement.keywords, query2refinement.preds.apply(lambda x: eval(x)[3])))

qu_df_temp['refinement_logic'] = qu_df_temp.progress_apply(lambda x: get_similar_refinements(x['PT'], 
                                                                                             x['Question String'], 
                                                                                             x['Picker Option Strings'], 
                                                                                             x['generated_keyword']), axis=1)

  7%|▋         | 94/1377 [13:55<3:41:33, 10.36s/it]

In [ ]:
qu_df_temp['refinement'] = qu_df_temp.progress_apply(lambda x: ';'.join(
    get_similar_refinements_(x['generated_keyword'], x['refinement_logic'])), axis=1)

In [ ]:
qu_df_temp['refinement_name'] = qu_df_temp['refinement'].apply(lambda x: bn2name.get(x, x))

In [ ]:
qu_df_temp['recommended_node_name'] = qu_df_temp['recommended_node'].apply(lambda x: bn2name.get(x, x))

In [185]:
qu_df_temp['is_multiselect'] = qu_df_temp['is_multiselect'].apply(lambda x: x.strip())
qu_df_temp['is_multiselect'].value_counts()

No     692
Yes    685
Name: is_multiselect, dtype: int64

In [187]:
glance = pd.read_csv("s3://ajjagata/2024/data_dumps/glance/EG/v2/part-00000-16c9aba7-c396-405b-8006-1d6d99f7538b-c000.csv")
glance = glance.rename(columns={'glance_views' : 'total_glance_views'})
glance['total_glance_views'] = (glance['total_glance_views'])#.apply(lambda x: min(x, 1))

In [188]:
refinement_level = pd.read_parquet("s3://ajjagata/2024/data_dumps/refinement_level/EG/v2/")

In [189]:
refinement_node_level = pd.read_parquet("s3://ajjagata/2024/data_dump/refinement_node_usage/EG/v2/")

In [190]:
refinement_level['refinement_id'] = refinement_level['refinement_id'].astype(str)
refinement_node_level['refinement_id'] = refinement_node_level['refinement_id'].astype(str)
refinement_node_level['picker_id'] = refinement_node_level['picker_id'].astype(str)

In [192]:
refinement_level_glance = refinement_level.merge(glance, on='product_type')
refinement_level_glance['perc_glance_views'] = refinement_level_glance['glance_views'] / refinement_level_glance['total_glance_views']

In [194]:
for threshold in [20, 30, 40, 50]:
    refinement_level_glance[f'fill_rate_satisfied_{threshold}'] = 'N'
    refinement_level_glance.loc[(refinement_level_glance['perc_glance_views'] > threshold/100), f'fill_rate_satisfied_{threshold}'] = 'Y'

In [195]:
refinement_node_level['asin_count_satisfied'] = 'N'
refinement_node_level.loc[refinement_node_level['asin_counts'] > 15, 'asin_count_satisfied'] = 'Y'

In [196]:
refinement_level_glance_req = refinement_level_glance[['product_type', 'refinement_id'] + [f'fill_rate_satisfied_{i}' for i in [20, 30, 40, 50]]]

In [197]:
refinement_node_level_req = refinement_node_level[['product_type', 'refinement_id', 'picker_id', 'asin_count_satisfied']]

In [198]:
refinement_checker = refinement_node_level_req.merge(refinement_level_glance_req, on=['product_type', 'refinement_id'], how='left')
refinement_checker = refinement_checker[['product_type', 'picker_id', 'asin_count_satisfied'] + [f'fill_rate_satisfied_{i}' for i in [20, 30, 40, 50]]]
refinement_checker = refinement_checker.rename(columns={'product_type' : 'PT', 'picker_id' : 'refinement'})

In [202]:
from collections import Counter

# Custom function to get the most popular item in a list
def most_popular_item(series):
    lst = [i for i in series.tolist() if not isinstance(i, type(None))]
    
    if len(lst) == 0:
        return None
    
    count = Counter(lst)
    max_count = max(count.values())
    # Return the most popular item. In case of tie, return the first item with the max count.
    return [item for item, cnt in count.items() if cnt == max_count][0]

multi_select_rows = qu_df_temp[qu_df_temp['is_multiselect'] == 'Yes']

# Group by 'PT' and 'Question String', and find the mode of 'recommended_node'
popular_nodes = multi_select_rows.groupby(['PT', 'Question String'])['recommended_node'].progress_apply(most_popular_item)

100%|██████████| 150/150 [00:00<00:00, 41013.40it/s]


In [205]:
# Update 'recommended_node' in the original DataFrame based on the mode values
for idx, row in popular_nodes.iteritems():
    pt, question_string = idx
    qu_df_temp.loc[(qu_df_temp['PT'] == pt) & (
        qu_df_temp['Question String'] == question_string) & (
        qu_df_temp['is_multiselect'] == 'Yes'), 'recommended_node'] = row

/tmp/ipykernel_28205/3580186906.py:2: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for idx, row in popular_nodes.iteritems():


In [206]:
qu_df_temp['recommended_node_name'] = qu_df_temp['recommended_node'].apply(lambda x: bn2name.get(x, x))

In [207]:
workflow = qu_df_temp

In [208]:
workflow.loc[workflow['Question Code'] == '1', 'Reformulation_Final'] = workflow.loc[workflow['Question Code'] == '1', 'generated_keyword']

In [209]:
workflow['Question Code'] = workflow['Question Code'].astype(str)

In [211]:
ancestory = workflow[['PT', 'Question Code', 'Next Question Code', 'recommended_node']]
ancestory['Next Question Code'] = ancestory['Next Question Code'].fillna("").apply(lambda x: x.split(','))
ancestory = ancestory.explode('Next Question Code')
ancestory = ancestory[ancestory['Next Question Code'] != '']
ancestory = ancestory.groupby(['PT', 'Next Question Code'])[['Question Code', 'recommended_node']].agg(
    lambda x: [i for i in x]).reset_index()

/tmp/ipykernel_28205/3796587242.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ancestory['Next Question Code'] = ancestory['Next Question Code'].fillna("").apply(lambda x: x.split(','))


In [212]:
def build_ancestors_map(df):
    ancestors_map = {}
    
    # Convert dataframe to a dictionary for quicker lookup
    children_dict = {}
    for _, row in tqdm(df.iterrows(), total=len(df)):
        child = row['browse_node_id']
        parent = row['parent_browse_node_id']
        if parent not in children_dict:
            children_dict[parent] = []
        children_dict[parent].append(child)
    
    # Depth-First Search to build ancestors_map
    def dfs(node):
        if node not in ancestors_map:
            ancestors_map[node] = set()
        for parent in children_dict.get(node, []):
            if parent not in ancestors_map:
                ancestors_map[parent] = set()
            ancestors_map[parent].add(node)
            ancestors_map[parent].update(ancestors_map[node])
            dfs(parent)
    
    # Start DFS from the root node(s)
    root_nodes = set(df['parent_browse_node_id']) - set(df['browse_node_id'])
    for root in root_nodes:
        dfs(root)
    
    return ancestors_map

def deepest_common_ancestor(ancestors_map, node_ids):    
    # Find common ancestors for all node_ids
    common_ancestors = set(ancestors_map[node_ids[0]])
    for node in node_ids[1:]:
        common_ancestors.intersection_update(ancestors_map[node])
    
    # Find the deepest common ancestor
    deepest_common_ancestor = max(common_ancestors, key=lambda x: len(ancestors_map[x]))
    
    return deepest_common_ancestor

In [213]:
node_ancestory = nodes[['browse_node_id', 'parent_browse_node_id']].astype(str)
ancestors_map = build_ancestors_map(node_ancestory)

100%|██████████| 298959/298959 [00:12<00:00, 24422.23it/s]


In [214]:
for node in ancestors_map:
    ancestors_map[node].add(node)

In [215]:
ancestory_grouped = ancestory.groupby(['PT', 'Next Question Code'])[['Question Code', 'recommended_node']].agg(
    lambda x: [i for i in x]).reset_index()

In [216]:
# ancestory_grouped['ancestor_node'] = ancestory_grouped['recommended_node'].progress_apply(
#     lambda x: deepest_common_ancestor(ancestors_map, x))

In [217]:
ancestory_grouped_required = ancestory_grouped[['PT', 'Next Question Code', 'recommended_node']].rename(
    columns={'Next Question Code' : 'Question Code', 'recommended_node' : 'parent_question_nodes'})

In [218]:
ancestory_re = workflow[['PT', 'Question Code', 'Next Question Code', 'recommended_node']]
ancestory_re = ancestory_re.merge(ancestory_grouped_required, on=['PT', 'Question Code'], how='left')

In [219]:
# workflow[workflow['recommended_node'].isna()]

In [220]:
def check_descendant(ancestors_map, node_ids, check_node):
    try:
        # print(node_ids, check_node)
        if isinstance(node_ids, float):
            return True
        for node in node_ids[0]:
            if node not in ancestors_map[check_node]:
                return False
        return True
    except:
        return False

In [221]:
ancestory_re['valid'] = ancestory_re.progress_apply(lambda x: check_descendant(
    ancestors_map, x['parent_question_nodes'], x['recommended_node']), axis=1)

100%|██████████| 1377/1377 [00:00<00:00, 119233.61it/s]


In [222]:
workflow['browse_node_valid'] = ancestory_re['valid'].apply(lambda x: 'Y' if x else 'N')

In [224]:
picker2refinement = dict(zip(nodes['browse_node_id'], nodes['parent_browse_node_id']))

In [225]:
refinement_valid_workflow = workflow.merge(refinement_checker, on=['PT', 'refinement'], how='left')
refinement_valid_workflow['browse_node_valid'] = ancestory_re['valid'].apply(lambda x: 'Y' if x else 'N')
for i in [f'fill_rate_satisfied_{i}' for i in [20, 30, 40, 50]]:
    refinement_valid_workflow[i] = refinement_valid_workflow[i].fillna('')
refinement_valid_workflow['asin_count_satisfied'] = refinement_valid_workflow['asin_count_satisfied'].fillna('')

In [226]:
final_workflow_output = refinement_valid_workflow[['PT', 'Question Code', 'Question String', 'Picker Option Strings', 'is_multiselect', 'Next Question Code', 
            'generated_keyword', 'Reformulation_Final', 'recommended_node', 'recommended_node_name', 'browse_node_valid', 'refinement', 
            'refinement_name', 'asin_count_satisfied'] + [f'fill_rate_satisfied_{i}' for i in [20, 30, 40, 50]]]

In [227]:
for column in ['recommended_node_name', 'recommended_node', 'refinement', 
               'refinement_name', 'asin_count_satisfied',
               'fill_rate_satisfied_20', 'fill_rate_satisfied_30',
               'fill_rate_satisfied_40', 'fill_rate_satisfied_50']:
    final_workflow_output.loc[final_workflow_output['browse_node_valid'] == 'N', column] = ''

/tmp/ipykernel_28205/3552784634.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_workflow_output.loc[final_workflow_output['browse_node_valid'] == 'N', column] = ''


In [228]:
for column in ['recommended_node', 'refinement', 
               'refinement_name', 'asin_count_satisfied',
               'fill_rate_satisfied_20', 'fill_rate_satisfied_30',
               'fill_rate_satisfied_40', 'fill_rate_satisfied_50', 'recommended_node_name']:
    final_workflow_output.loc[final_workflow_output['recommended_node_name'].apply(lambda x: "us-books" in str(x)), column] = ''

/tmp/ipykernel_28205/2673451402.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_workflow_output.loc[final_workflow_output['recommended_node_name'].apply(lambda x: "us-books" in str(x)), column] = ''


In [229]:
final_workflow_output['picker'] = final_workflow_output['refinement']
final_workflow_output['refinement'] = final_workflow_output['picker'].apply(lambda x: picker2refinement.get(x, x))

/tmp/ipykernel_28205/164798443.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_workflow_output['picker'] = final_workflow_output['refinement']
/tmp/ipykernel_28205/164798443.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_workflow_output['refinement'] = final_workflow_output['picker'].apply(lambda x: picker2refinement.get(x, x))


In [230]:
final_workflow_output.loc[final_workflow_output['fill_rate_satisfied_20'] == '', 'picker'] = ''
final_workflow_output.loc[final_workflow_output['fill_rate_satisfied_20'] == '', 'refinement'] = ''
final_workflow_output.loc[final_workflow_output['fill_rate_satisfied_20'] == '', 'refinement_name'] = ''

In [231]:
workflow_base = pd.read_excel("workflows/MENA_final_workflows.xlsx")

In [ ]:
final_workflow_output = final_workflow_output[['PT', 'Question Code', 'Question String', 'Picker Option Strings',
       'is_multiselect', 'Next Question Code', 'generated_keyword',
       'Reformulation_Final', 'generated_keyword_ar', 'Reformulation_Final_ar', 'recommended_node', 'recommended_node_name',
       'browse_node_valid', 'refinement', 'refinement_name',
       'asin_count_satisfied', 'fill_rate_satisfied_20',
       'fill_rate_satisfied_30', 'fill_rate_satisfied_40',
       'fill_rate_satisfied_50', 'picker']]

/tmp/ipykernel_28205/3875119953.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_workflow_output['generated_keyword_ar'] = mena_arabic['generated_keyword']
/tmp/ipykernel_28205/3875119953.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_workflow_output['Reformulation_Final_ar'] = mena_arabic['Reformulation_Final']


In [233]:
final_workflow_output = final_workflow_output.rename(columns={'Question String' : 'Question', 
                                                              'Picker Option Strings' : 'options', 
                                                              'PT' : 'product_type'})

In [237]:
workflow_base_output = workflow_base.merge(final_workflow_output, on=['product_type', 'Question', 'options'], how='left')

In [238]:
refinement2short_id_df = pd.read_parquet("s3://ajjagata/2024/data_dumps/short_id/v0/")
refinement2short_id = dict(zip(refinement2short_id_df.node_id.astype(str), 'p_n_' + refinement2short_id_df.refinement_bin.astype(str)))
workflow_base_output['refinement_bin'] = workflow_base_output['refinement'].apply(lambda x: refinement2short_id.get(x, ''))

In [242]:
workflow_base_output.to_excel(
    'workflow_final/mapped_output_EG_v11.xlsx', index=False)